# 02 — M1 baseline (TV1, Việc 5 + Việc 6)

**Add Input:** Kaggle Dataset `nhom05-falldet-processed`. Bật **GPU** và **Internet** (tải trọng số YOLO11 lần đầu).

Luồng: tạo tracks tạm → tune trên **val** → chạy M1 → đánh giá trên **test** của URFD và Le2i → đánh giá cross-dataset.
Ngưỡng chỉ chọn trên val, không bao giờ dùng test.

In [ ]:
# Ô đầu tiên: lấy mã nguồn, cố định seed, in version Python, torch, ultralytics, opencv và version dataset.
# Cần bật Internet để clone repo.
import os, sys
BRANCH = "tv1/gk-data-eda-m1"
REPO = "/kaggle/working/FallDetection"
if not os.path.exists(REPO):
    !git clone --depth 1 -b {BRANCH} https://github.com/ducqwang/FallDetection.git {REPO}
%cd {REPO}
!pip install -q ultralytics==8.4.202 lap==0.5.14
sys.path.insert(0, REPO)
from src.data.env_info import setup_notebook
setup_notebook("configs/data.yaml", seed=42)

In [ ]:
from pathlib import Path
from src.data.loader import data_root
ROOT = data_root()
META = f"{ROOT}/metadata.csv"
GT = f"{ROOT}/annotations/le2i_gt_bbox.csv"
WORK = Path("/kaggle/working/m1")
WORK.mkdir(exist_ok=True)
TRACKS = WORK / "tracks.parquet"
RESULTS = "reports/results"
print("Dữ liệu:", ROOT)

## 1. Tạo tracks tạm

`scripts/make_tracks_tmp.py` là bản **tạm thời**. Khi TV2 có tracks chính thức, Add Input file đó, trỏ `TRACKS` vào và bỏ qua ô này.

In [ ]:
!python scripts/make_tracks_tmp.py --meta {META} --out {TRACKS} --model yolo11n.pt

## 2. Tune trên tập val

In [ ]:
!python scripts/tune_m1.py --tracks {TRACKS} --meta {META} --dataset urfd --out {RESULTS}/m1_tuned_urfd.yaml
!python scripts/tune_m1.py --tracks {TRACKS} --meta {META} --dataset le2i --out {RESULTS}/m1_tuned_le2i.yaml
!python scripts/tune_m1.py --gt-bbox {GT} --meta {META} --dataset le2i --out {RESULTS}/m1_tuned_le2i_gt.yaml

## 3. Chạy M1 với ngưỡng đã tune

In [ ]:
!python -m src.models.m1_rule --tracks {TRACKS} --meta {META} --tuned {RESULTS}/m1_tuned_urfd.yaml --out {WORK}/m1_urfd.csv
!python -m src.models.m1_rule --tracks {TRACKS} --meta {META} --tuned {RESULTS}/m1_tuned_le2i.yaml --out {WORK}/m1_le2i.csv
!python -m src.models.m1_rule --gt-bbox {GT} --meta {META} --tuned {RESULTS}/m1_tuned_le2i_gt.yaml --out {WORK}/m1_le2i_gt.csv

## 4. Đánh giá trên tập test (T = 0.5, 1.0, 2.0)

Le2i GT là **kết quả cận trên**: so với dòng dùng tracks để tách lỗi của mô hình với lỗi của detector.
Mỗi bảng có mục ADL negative khó (`adl_hard_neg = 1`, chỉ có ở URFD).

In [ ]:
!python scripts/evaluate.py --pred {WORK}/m1_urfd.csv --meta {META} --split test --dataset urfd --tuned {RESULTS}/m1_tuned_urfd.yaml --T 0.5 1.0 2.0 --tag m1_tracks
!python scripts/evaluate.py --pred {WORK}/m1_le2i.csv --meta {META} --split test --dataset le2i --tuned {RESULTS}/m1_tuned_le2i.yaml --T 0.5 1.0 2.0 --tag m1_tracks
!python scripts/evaluate.py --pred {WORK}/m1_le2i_gt.csv --meta {META} --split test --dataset le2i --tuned {RESULTS}/m1_tuned_le2i_gt.yaml --T 0.5 1.0 2.0 --tag m1_gt

## 5. Cross-dataset

Le2i chỉ có khoảng 5 ADL trong tập test nên kết quả within-dataset nhiễu; ưu tiên báo cáo các dòng này.
Protocol A: ngưỡng tune trên URFD → toàn bộ 130 video Le2i. Protocol B: ngược lại → toàn bộ 70 video URFD.

In [ ]:
!python scripts/evaluate.py --pred {WORK}/m1_urfd.csv --meta {META} --split all --dataset le2i --tuned {RESULTS}/m1_tuned_urfd.yaml --tag m1_cross_A
!python scripts/evaluate.py --pred {WORK}/m1_le2i.csv --meta {META} --split all --dataset urfd --tuned {RESULTS}/m1_tuned_le2i.yaml --tag m1_cross_B

## 6. Bảng tổng hợp

In [ ]:
!python scripts/summarize_results.py --results-dir {RESULTS} --out {RESULTS}/m1_summary.csv

Tải `reports/results/*` về và commit lên nhánh.